# 5G Network Intelligence — Clipped Forecast Blending

## Hypothesis and evidence

Nonnegative direct training reduces zero-target error but often increases positive-target error. Its residual correlation with hurdle is approximately 0.93–0.95, which does not prove blending will help. Test a bounded convex blend and retain weight zero. No base model is retrained.


In [ ]:
# Choosing coefficients is calibration: hosted execution only.
RUN_SELECTION = False


## Temporal protocol and acceptance

Select each horizon's weight from 0/0.05/0.10/0.20 on inner1 only, ties favor smaller weights. Persist all choices before reading inner2 numeric forecasts. Apply unchanged on inner2. Require no horizon regression against hurdle or the previously confirmed MLP-B/hurdle blend, and at least 0.1% relative MAE gain at one horizon with nonzero weight. Do not adjust weights after confirmation. Both folds are reused development evidence. Validation 504–671, final holdout and production are excluded.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
# Review saved outputs without fitting or selection.
PROTOCOL_FILE = Path('configs/clipped_blend.yaml')
if PROTOCOL_FILE.exists():
    import yaml
    print(yaml.safe_load(PROTOCOL_FILE.read_text()))


## Immutable sources and preprocessing

Use saved direct-clipped forecasts, original LightGBM hurdle forecasts and the previous MLP blend confirmation. Source manifest hashes are frozen in the config. Every source file is verified; actual/origin/beam axes must match exactly. No new preprocessing, raw panel access or prediction-threshold fitting is needed.


## Cloud execution controls

The dedicated worker attaches three completed private source jobs. Its setup verifies the source package and locates pinned artifact roots. Review Run All skips all calibration. Checkpoints remain immutable and new selection outputs stay in Kaggle Output.


In [ ]:
if RUN_SELECTION:
    import subprocess, sys, tarfile
    command = [sys.executable, '-u', '-m', 'scripts.select_clipped_blend',
               '--clipped-root', str(CLIPPED), '--hurdle-root', str(HURDLE), '--current-root', str(CURRENT)]
    subprocess.run(command, check=True)
    with tarfile.open('/kaggle/working/clipped_blend_results.tar.gz', 'w:gz') as archive:
        for folder in ('artifacts/clipped_blend', 'reports'):
            archive.add(folder, arcname=folder)
else:
    print('Review mode: coefficient selection skipped.')


## Evaluation and limitations

Report every horizon, zero-weight choices and regressions. MAE is in original DLPRB units; WAPE is sum absolute error / sum absolute actual. A development pass does not authorize validation refitting or establish unseen-test superiority. A strict-gate failure ends this recipe.


In [ ]:
for name in ('clipped_blend_selection_grid.csv', 'clipped_blend_confirmation.csv', 'clipped_blend_current_control.csv'):
    path = Path('reports')/name
    if path.exists():
        display(pd.read_csv(path))


## Completed and verified result

The selected clipped weight was 0.20 at every horizon. Saved forecasts replay exactly. Hurdle-only gains confirm, but the strict current-control gate fails: +24h regresses and the +1h/+6h gains remain below 0.1%. Reject this recipe; no coefficient changes, validation refit or production promotion. Evidence: `reports/clipped_blend_analysis.md`.
